# Deep Agents 하이브리드 에이전트 — RAG + MCP + Skills

`create_deep_agent`로 세 가지 핵심 기술을 하나의 에이전트에 통합합니다.

| 시나리오 | 기대 동작 | 사용 기술 |
|----------|----------|----------|
| 회사 사내 규정 질문 | RAG 벡터 검색 → 문서 기반 답변 | `retrieve` 도구 |
| 수학 계산 요청 | MCP 서버 호출 → 계산 결과 | `add`, `multiply` 도구 |
| 보고서 작성 요청 | SKILL.md 양식 로드 → 양식에 맞는 출력 | `skills=` 파라미터 |

Langfuse로 각 시나리오의 트레이스를 기록하고, **어떤 도구가 호출되었는지** 검증합니다.

In [ ]:
# 환경 설정
import os, logging
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

model = ChatOpenAI(model="gpt-5.4-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("환경 준비 완료.")

In [ ]:
# Langfuse 트레이싱 설정
from langfuse.langchain import CallbackHandler

langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]}
print(f"Langfuse ON — {os.environ.get('LANGFUSE_BASE_URL', '')}")

---
## Step 1. RAG — 회사 사내 규정 지식 베이스

회사 HR 규정 문서를 벡터 스토어에 인덱싱하고, `retrieve` 도구로 검색합니다.